# 01 - Explore the dataset

**Question.** What does the feature space look like, and are the classes balanced
enough to train on?

**Author.** abc &nbsp;|&nbsp; **Status.** exploration

**Discipline.** This notebook contains *narrative and inspection only*. Every
function it calls lives in `src/`, is typed, and is covered by a test. If you
find yourself writing a function in a cell, move it to `src/` before it grows a
second caller.

**Findings.** _(fill in once the analysis is done)_

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

# --- Setup: import reusable code from src/, never redefine it here -----------
# `whet init` cannot substitute the package name into .ipynb JSON, so the package
# is resolved from the source tree instead of hard-coded. In notebooks you write
# yourself, replace the import_module lines with `from your_package import ...`.
import sys
from importlib import import_module
from pathlib import Path

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PACKAGE = next(p.name for p in sorted(SRC.iterdir()) if (p / "__init__.py").is_file())
config_mod = import_module(f"{PACKAGE}.config")
data_mod = import_module(f"{PACKAGE}.data")
viz_mod = import_module(f"{PACKAGE}.viz")

viz_mod.use_project_style()
PACKAGE, plt.get_backend()

In [ ]:
# --- Configuration: one validated object, no loose magic numbers -------------
cfg = config_mod.ExperimentConfig(
    name="explore-dataset",
    seed=42,
    n_samples=600,
    n_classes=3,
    notes="First look at the feature space and class balance.",
)
cfg.log_summary()
cfg

## Load the data

`make_synthetic_dataset` is a placeholder so this notebook runs on a fresh clone
with no download. Point it at your real loader in `src/` when you have one.

In [ ]:
features, labels = data_mod.make_synthetic_dataset(
    n_samples=cfg.n_samples,
    n_features=cfg.n_features,
    n_classes=cfg.n_classes,
    seed=cfg.seed,
)
features.shape, labels.shape

## Class balance

A severely imbalanced dataset invalidates plain accuracy before a single model is
trained, so this is the first thing to check.

In [ ]:
counts = data_mod.class_counts(labels)
balance = data_mod.class_balance(labels)
counts, round(balance, 3)

## Splits

Disjoint by construction — `split_indices` is tested for it.

In [ ]:
train_idx, val_idx, test_idx = data_mod.split_indices(
    n_samples=cfg.n_samples,
    train_fraction=cfg.train_fraction,
    val_fraction=cfg.val_fraction,
    seed=cfg.seed,
)
train_idx.size, val_idx.size, test_idx.size

## Figures

Cell outputs are stripped on commit by the `nbstripout` pre-commit hook. Anything
worth keeping is saved to `outputs/figures/` as a real file.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))
viz_mod.plot_class_distribution(labels, ax=left)
viz_mod.plot_scatter_2d(features[train_idx], labels[train_idx], ax=right, title="Train split")
fig.tight_layout()
saved = viz_mod.save_figure(fig, cfg.figure_path("overview"), close=False)
saved

## Conclusions and next steps

- _What did you learn?_
- _What is the next experiment?_
- _Which cell here has earned promotion into `src/`?_

Before committing: **Kernel > Restart & Run All**, then let the pre-commit hook
strip the outputs.